<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/pid_datasheet_to_cad_cfd_neqsim.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg"/></a>

# P&ID + mechanical datasheet → CadQuery → NeqSim → OpenFOAM

End-to-end educational example for a horizontal three-phase separator. Public references: Kimray's *How to Read an Oil & Gas P&ID Reference Guide* and the University of Oklahoma *Oil and Gas Separation Design Manual*. The dimensional datasheet below is synthetic and explicitly traceable; it does not reproduce proprietary vendor data.

In [ ]:
!pip -q install neqsim cadquery trimesh pyvista pandas

In [ ]:
from pathlib import Path
import json, math, shutil
import pandas as pd
import numpy as np
WORK=Path("/content/pid_datasheet_cfd"); WORK.mkdir(exist_ok=True)

## 1. Structured P&ID and datasheet

P&ID data provides tags, topology, line sizes and controls. Mechanical data provides dimensions and nozzle locations. A human-review confidence column prevents uncertain document extraction from silently becoming CFD geometry.

In [ ]:
model={
"references":{
 "pid":"https://kimray.com/sites/default/files/uploads/training-demos/Kimray%20How%20to%20Read%20an%20Oil%20%26%20Gas%20P%26ID%20Reference%20Guide.pdf",
 "design":"https://ou.edu/content/dam/pacs/laurance-reid/documents/resources-docs/3_oil_and_gas_separation_design_manual_by_c_richard_sivalls.pdf"},
"pid":{"tag":"20-VG-001","type":"horizontal three-phase separator",
 "lines":[["20-P-001","inlet",0.3048],["20-P-002","gas outlet",0.2540],["20-P-003","oil outlet",0.1016],["20-P-004","water outlet",0.0762]],
 "controls":["20-PIC-001","20-LIC-001","20-LIC-002"]},
"datasheet":{"status":"synthetic educational data","ID_m":2.4,"tangent_length_m":7.2,
 "head":"2:1 ellipsoidal","P_design_bara":85,"T_design_C":100,"P_oper_bara":65,"T_oper_C":45,
 "oil_level_m":0.85,"water_level_m":0.35,
 "nozzles":[["N1","inlet",0.3048,1.0,"top"],["N2","gas outlet",0.254,6.2,"top"],
            ["N3","oil outlet",0.1016,5.7,"bottom"],["N4","water outlet",0.0762,1.6,"bottom"]]}}
(WORK/"engineering_model.json").write_text(json.dumps(model,indent=2))
pd.DataFrame([
["Tag","20-VG-001","P&ID","high"],["ID","2.40 m","synthetic datasheet","medium"],
["Tangent length","7.20 m","synthetic datasheet","medium"],["Nozzles","N1–N4","P&ID + datasheet","high"],
["Internals","simplified","engineering assumption","low"]],
columns=["Parameter","Value","Source","Confidence"])

## 2. NeqSim thermodynamics and CFD inlet data

In [ ]:
from neqsim.thermo import fluid
f=fluid("cpa")
for c,z in {"nitrogen":0.7,"CO2":2.0,"methane":72,"ethane":7,"propane":5,
            "i-butane":1.5,"n-butane":2,"i-pentane":1,"n-pentane":1,
            "n-hexane":2,"n-heptane":2,"n-octane":1.5,"water":2.3}.items():
    f.addComponent(c,z)
f.setMixingRule(10); f.setMultiPhaseCheck(True)
f.setTemperature(45,"C"); f.setPressure(65,"bara"); f.setTotalFlowRate(120000,"kg/hr")
from neqsim.thermo import TPflash
TPflash(f); f.initProperties()
rows=[]
A=math.pi*model["pid"]["lines"][0][2]**2/4
for i in range(f.getNumberOfPhases()):
    p=f.getPhase(i); q=p.getFlowRate("m3/sec")
    rows.append({"phase":str(p.getPhaseTypeName()),"mass_kg_s":p.getFlowRate("kg/sec"),
      "q_m3_s":q,"rho_kg_m3":p.getDensity("kg/m3"),"mu_Pa_s":p.getViscosity("kg/msec"),
      "superficial_velocity_m_s":q/A})
phase_table=pd.DataFrame(rows); phase_table

## 3. CadQuery internal fluid domain

The CFD model is the internal volume. The geometry is parameterized directly from the reviewed datasheet and exported as STEP and STL.

In [ ]:
import cadquery as cq
d=model["datasheet"]; D=d["ID_m"]; L=d["tangent_length_m"]; R=D/2
domain=cq.Workplane("YZ").circle(R).extrude(L)
# Add simplified nozzle volumes
for tag,service,diam,x,pos in d["nozzles"]:
    z=R if pos=="top" else -R-0.8
    n=cq.Workplane("XY",origin=(x,0,z)).circle(diam/2).extrude(0.8)
    domain=domain.union(n)
step=WORK/"20-VG-001_fluid_domain.step"; stl=WORK/"20-VG-001_fluid_domain.stl"
cq.exporters.export(domain,str(step)); cq.exporters.export(domain,str(stl),tolerance=0.003)
print(step,stl)

In [ ]:
import trimesh, pyvista as pv
tri=trimesh.load_mesh(stl)
display(pd.DataFrame([{"watertight":tri.is_watertight,"volume_m3":abs(tri.volume),
 "area_m2":tri.area,"extents_m":tri.extents.tolist()}]))
mesh=pv.read(str(stl)); p=pv.Plotter(notebook=True); p.add_mesh(mesh,show_edges=True,opacity=.8)
p.show(jupyter_backend="static")

## 4. Generate an OpenFOAM-ready case

The notebook writes a portable `blockMeshDict`, `snappyHexMeshDict`, STL and NeqSim phase-property JSON. Solver selection and named boundary splitting must be reviewed before production CFD.

In [ ]:
case=WORK/"openfoam_case"
for pth in ["constant/triSurface","system","0"]: (case/pth).mkdir(parents=True,exist_ok=True)
shutil.copy2(stl,case/"constant/triSurface/separator.stl")
(case/"constant/neqsimBoundaryConditions.json").write_text(json.dumps({
 "pressure_bara":d["P_oper_bara"],"temperature_C":d["T_oper_C"],
 "oil_level_m":d["oil_level_m"],"water_level_m":d["water_level_m"],
 "phases":phase_table.to_dict("records")},indent=2))
(case/"system/blockMeshDict").write_text(f'''FoamFile{{format ascii;class dictionary;object blockMeshDict;}}
convertToMeters 1;
vertices((-1 -2 -2)({L+1.5} -2 -2)({L+1.5} 2 -2)(-1 2 -2)(-1 -2 2)({L+1.5} -2 2)({L+1.5} 2 2)(-1 2 2));
blocks(hex (0 1 2 3 4 5 6 7) (60 30 30) simpleGrading (1 1 1));
boundary(farfield{{type patch;faces((0 4 7 3)(1 2 6 5)(0 1 5 4)(3 7 6 2)(0 3 2 1)(4 5 6 7));}});''')
(case/"system/snappyHexMeshDict").write_text(f'''FoamFile{{format ascii;class dictionary;object snappyHexMeshDict;}}
castellatedMesh true;snap true;addLayers false;
geometry{{separator.stl{{type triSurfaceMesh;name separator;}}}}
castellatedMeshControls{{maxLocalCells 1000000;maxGlobalCells 2000000;minRefinementCells 10;
nCellsBetweenLevels 3;features();refinementSurfaces{{separator{{level (2 3);patchInfo{{type wall;}}}}}}
resolveFeatureAngle 30;refinementRegions{{}}locationInMesh ({L/2} 0 0);allowFreeStandingZoneFaces true;}}
snapControls{{nSmoothPatch 3;tolerance 2;nSolveIter 30;nRelaxIter 5;}}
addLayersControls{{}}meshQualityControls{{}}''')
print(*[str(x.relative_to(case)) for x in case.rglob("*") if x.is_file()],sep="\n")

## 5. CFD → NeqSim feedback contract

A detailed CFD campaign should return pressure loss, residence time and carryover/carryunder maps. These become fast reduced-order correlations in the full NeqSim process model.

In [ ]:
cfd_result={"status":"placeholder until OpenFOAM is run","deltaP_bar":0.18,
 "liquid_carryover_mass_fraction":0.0025,"oil_in_water_mass_fraction":0.006}
pd.DataFrame([
 ["NeqSim inlet pressure",d["P_oper_bara"],"bara"],
 ["CFD pressure loss",cfd_result["deltaP_bar"],"bar"],
 ["NeqSim outlet pressure",d["P_oper_bara"]-cfd_result["deltaP_bar"],"bara"],
 ["Liquid carryover",100*cfd_result["liquid_carryover_mass_fraction"],"mass %"]],
 columns=["Quantity","Value","Unit"])

In [ ]:
archive=shutil.make_archive(str(WORK/"pid_datasheet_neqsim_cfd_artifacts"),"zip",WORK)
print(archive)

## Conclusions

This digital thread preserves the same equipment and stream tags from documentation through NeqSim, CAD and CFD. Replace the synthetic datasheet with approved project data, review all low-confidence extraction, create named boundary patches, select the appropriate multiphase solver, and fit CFD response surfaces back into NeqSim for process-wide studies.